# Baseline 1 — TabTransformer on ISCX-URL2016
**Mục tiêu:** 29 tabular features → TabTransformer → binary classification  
**Dataset:** `iscxurl2016` → `ISCXURL2016.csv`  
**Thời gian:** ~15–30 phút trên GPU T4

In [ ]:
import os, sys, json, re, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)
from tqdm.notebook import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
PROJECT = Path('..')
RAW_DIR = PROJECT / 'data' / 'raw'
OUT_DIR = Path('/kaggle/working') if KAGGLE_INPUT.exists() else PROJECT / 'data'
MODEL_DIR = OUT_DIR / 'models'; FIG_DIR = OUT_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect ISCX URL2016 CSV anywhere under /kaggle/input/
iscx_csv = RAW_DIR / 'ISCXURL2016.csv'
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.rglob('ISCXURL2016.csv'):
        iscx_csv = p; break
print(f'ISCX: {iscx_csv.exists()} -> {iscx_csv}')

ISCX: True -> /kaggle/input/datasets/haidang07091998/iscxurl2016/ISCXURL2016.csv


In [ ]:
ISCX_FEATURES_NUM = [
    'urlLen','domainlength','pathLength','subDirLen','fileNameLen',
    'this.fileExtLen','ArgLen','Entropy_URL','Entropy_Domain',
    'Entropy_DirectoryName','Entropy_Filename','Entropy_Afterpath',
    'spcharUrl','URL_DigitCount','host_DigitCount','NumberRate_URL',
    'NumberRate_Domain','NumberRate_DirectoryName','NumberRate_FileName',
    'SymbolCount_URL','SymbolCount_Domain','URL_Letter_Count',
    'host_letter_count','NumberofDotsinURL','LongestPathTokenLength',
    'CharacterContinuityRate','Domain_LongestWordLength',
]
ISCX_FEATURES_CAT = ['URL_sensitiveWord', 'ISIpAddressInDomainName']
TABULAR_DIM = 29

df = pd.read_csv(iscx_csv, encoding='utf-8', low_memory=False)
print(f'Rows: {len(df)}, Cols: {len(df.columns)}')

avail_num = [c for c in ISCX_FEATURES_NUM if c in df.columns]
avail_cat = [c for c in ISCX_FEATURES_CAT if c in df.columns]
print(f'Numerical: {len(avail_num)}/{len(ISCX_FEATURES_NUM)}, Categorical: {len(avail_cat)}/{len(ISCX_FEATURES_CAT)}')

# Numerical
X_num = df[avail_num].copy().replace([np.inf, -np.inf], np.nan)
for c in X_num.columns: X_num[c] = pd.to_numeric(X_num[c], errors='coerce')
X_num = X_num.fillna(0).astype(np.float32)

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num).astype(np.float32)

# Categorical
X_cat = df[avail_cat].fillna(0).astype(int).clip(lower=0) if avail_cat else None

# Combine
if X_cat is not None and len(avail_cat) > 0:
    X = np.concatenate([X_num_scaled, X_cat.values.astype(np.float32)], axis=1)
else:
    X = X_num_scaled

# Pad to 29
if X.shape[1] < TABULAR_DIM:
    pad = np.zeros((len(df), TABULAR_DIM - X.shape[1]), dtype=np.float32)
    X = np.concatenate([X, pad], axis=1)
elif X.shape[1] > TABULAR_DIM:
    X = X[:, :TABULAR_DIM]

y = (df['URL_Type_obf_Type'].astype(str).str.strip().str.lower() == 'phishing').astype(int).values
print(f'Phishing: {y.sum()}, Benign: {(y==0).sum()}')
print(f'Shape: {X.shape}')


Rows: 36707, Cols: 80
Numerical: 27/27, Categorical: 2/2
Phishing: 7586, Benign: 29121
Shape: (36707, 29)


In [ ]:
class FeatureEmbedding(nn.Module):
    def __init__(self, d=32): super().__init__(); self.e = nn.Linear(1, d)
    def forward(self, x): return self.e(x.unsqueeze(-1))

class TabTransformer(nn.Module):
    def __init__(self, nf=29, ed=32, nh=4, hd=256, od=128, dp=0.1):
        super().__init__()
        self.embs = nn.ModuleList([FeatureEmbedding(ed) for _ in range(nf)])
        self.attn = nn.MultiheadAttention(ed, nh, batch_first=True, dropout=dp)
        self.n1 = nn.LayerNorm(ed); self.n2 = nn.LayerNorm(ed)
        self.ff = nn.Sequential(nn.Linear(ed, hd), nn.GELU(), nn.Dropout(dp), nn.Linear(hd, ed), nn.Dropout(dp))
        self.proj = nn.Linear(ed * nf, od)
        self.cls = nn.Sequential(nn.Linear(od, 64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 1))
    def forward(self, x):
        h = torch.stack([e(x[:, i]) for i, e in enumerate(self.embs)], 1)
        a, _ = self.attn(h, h, h); h = self.n1(h + a)
        f = self.ff(h); h = self.n2(h + f)
        return self.cls(self.proj(h.reshape(h.size(0), -1)))

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X) if isinstance(X, np.ndarray) else X
        self.y = torch.from_numpy(y.reshape(-1,1).astype(np.float32)) if isinstance(y, np.ndarray) else y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def compute_metrics(labels, preds):
    pb = (preds >= 0.5).astype(int)
    fpr_val = 0.0
    if len(np.unique(labels)) > 1:
        cm  = confusion_matrix(labels, pb)
        tn, fp = cm[0, 0], cm[0, 1]
        fpr_val = round(fp / max(tn + fp, 1), 4)
    return {'accuracy': accuracy_score(labels, pb),
            'precision': precision_score(labels, pb, zero_division=0),
            'recall': recall_score(labels, pb, zero_division=0),
            'f1': f1_score(labels, pb, zero_division=0),
            'auc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else 0.0,
            'fpr': fpr_val}

def train_epoch(model, loader, opt, crit):
    model.train(); total = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = crit(model(Xb), yb)
        loss.backward(); opt.step(); total += loss.item() * Xb.size(0)
    return total / len(loader.dataset)

def evaluate(model, loader, crit):
    model.eval(); total = 0; preds, labs = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            total += crit(logits, yb).item() * Xb.size(0)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labs.extend(yb.cpu().numpy())
    preds, labs = np.array(preds), np.array(labs)
    m = compute_metrics(labs, preds); m['loss'] = total / len(loader.dataset)
    return m, preds, labs

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_metrics, all_preds, all_labels = [], [], []
BS, EP, LR = 64, 50, 1e-3

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    tr_ld = DataLoader(SimpleDataset(X_tr, y_tr), batch_size=BS, shuffle=True)
    te_ld = DataLoader(SimpleDataset(X_te, y_te), batch_size=BS)
    model = TabTransformer(nf=X.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
    crit = nn.BCEWithLogitsLoss()
    for ep in range(1, EP + 1):
        tl = train_epoch(model, tr_ld, opt, crit)
        m, _, _ = evaluate(model, te_ld, crit)
        sched.step()
        if ep % 10 == 0:
            print(f'  Epoch {ep:2d}/{EP} | Loss: {tl:.4f} | AUC: {m["auc"]:.4f} | F1: {m["f1"]:.4f}')
    fm, fp, fl = evaluate(model, te_ld, crit)
    fm['fold'] = fold + 1
    all_metrics.append(fm); all_preds.append(fp); all_labels.append(fl)
    torch.save(model.state_dict(), MODEL_DIR / f'baseline1_fold{fold+1}.pt')
    print(f'  Done: Acc={fm["accuracy"]:.4f}, AUC={fm["auc"]:.4f}, F1={fm["f1"]:.4f}')

avg = {k: np.mean([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
std = {k: np.std([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
print(f'\n>>> 5-Fold CV: Acc={avg["accuracy"]:.4f}+-{std["accuracy"]:.4f}, AUC={avg["auc"]:.4f}+-{std["auc"]:.4f}, F1={avg["f1"]:.4f}+-{std["f1"]:.4f}')


--- Fold 1/5 ---
  Epoch 10/50 | Loss: 0.0494 | AUC: 0.9925 | F1: 0.9591
  Epoch 20/50 | Loss: 0.0095 | AUC: 0.9976 | F1: 0.9761
  Epoch 30/50 | Loss: 0.0034 | AUC: 0.9976 | F1: 0.9769
  Epoch 40/50 | Loss: 0.0016 | AUC: 0.9984 | F1: 0.9796
  Epoch 50/50 | Loss: 0.0013 | AUC: 0.9984 | F1: 0.9796
  Done: Acc=0.9898, AUC=0.9984, F1=0.9796

--- Fold 2/5 ---
  Epoch 10/50 | Loss: 0.0432 | AUC: 0.9945 | F1: 0.9618
  Epoch 20/50 | Loss: 0.0091 | AUC: 0.9973 | F1: 0.9711
  Epoch 30/50 | Loss: 0.0035 | AUC: 0.9985 | F1: 0.9804
  Epoch 40/50 | Loss: 0.0017 | AUC: 0.9988 | F1: 0.9812
  Epoch 50/50 | Loss: 0.0015 | AUC: 0.9985 | F1: 0.9808
  Done: Acc=0.9914, AUC=0.9988, F1=0.9812

--- Fold 3/5 ---
  Epoch 10/50 | Loss: 0.0445 | AUC: 0.9941 | F1: 0.9614
  Epoch 20/50 | Loss: 0.0087 | AUC: 0.9978 | F1: 0.9734
  Epoch 30/50 | Loss: 0.0035 | AUC: 0.9980 | F1: 0.9754
  Epoch 40/50 | Loss: 0.0016 | AUC: 0.9985 | F1: 0.9812
  Epoch 50/50 | Loss: 0.0012 | AUC: 0.9986 | F1: 0.9816
  Done: Acc=0.9920, AU

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']

all_p = np.concatenate(all_preds); all_l = np.concatenate(all_labels)
cm = confusion_matrix(all_l, (all_p >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
axes[0].set_title('Baseline 1 — Confusion Matrix'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

for f in range(N_FOLDS):
    fpr, tpr, _ = roc_curve(all_labels[f], all_preds[f])
    axes[1].plot(fpr, tpr, color=colors[f], lw=1.5, alpha=0.7,
                 label=f'Fold {f+1} (AUC={roc_auc_score(all_labels[f], all_preds[f]):.4f})')
fpr, tpr, _ = roc_curve(all_l, all_p)
axes[1].plot(fpr, tpr, 'k--', lw=2.5, label=f'Mean (AUC={roc_auc_score(all_l, all_p):.4f})')
axes[1].plot([0,1],[0,1], 'gray', lw=1, alpha=0.5)
axes[1].set_title('ROC Curves'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8, loc='lower right')

names = ['accuracy','precision','recall','f1','auc']
x = np.arange(len(names)); means = [avg[m] for m in names]; stdevs = [std[m] for m in names]
axes[2].bar(x, means, yerr=stdevs, capsize=5, color='#1f77b4', alpha=0.8)
axes[2].set_xticks(x); axes[2].set_xticklabels([m.capitalize() for m in names])
axes[2].set_ylim(0, 1); axes[2].set_title('Metrics (Mean+-Std)')
for i, (m, s) in enumerate(zip(means, stdevs)):
    axes[2].text(i, m + s + 0.02, f'{m:.3f}+-{s:.3f}', ha='center', fontsize=8)

plt.tight_layout(); plt.savefig(FIG_DIR / 'baseline1_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "baseline1_summary.png"}')

Saved: ../data/figures/baseline1_summary.png


In [ ]:
results = {'model': 'Baseline 1 - TabTransformer (ISCX)', **avg, **{k + '_std': float(std[k]) for k in std}}
with open(MODEL_DIR / 'evaluation_baseline1.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {MODEL_DIR / "evaluation_baseline1.json"}')

Results saved to ../data/models/evaluation_baseline1.json


---
### Download từ Output tab:
- `figures/baseline1_summary.png`
- `data/models/evaluation_baseline1.json`  
- `data/models/baseline1_fold1..5.pt` (optional, để load sau)

Sau đó chạy **Baseline 2** → `kaggle_baseline2.ipynb`